### Raw data

In [1]:
import numpy as np
import pandas as pd

from scipy.signal import butter, filtfilt, welch
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression 
# from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix


In [2]:
FS = 500  # sampling rate
LOW, HIGH = 0.5, 45
EPOCH_SEC = 4
OVERLAP = 0.5

BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 25),
    "gamma": (25, 45),
}

In [3]:
def bandpass_filter(signal, fs, low, high, order=4):
    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype="band")
    return filtfilt(b, a, signal, axis=-1)

In [4]:
def epoch_signal(signal, fs, epoch_sec, overlap):

    step = int(epoch_sec * fs * (1 - overlap))
    size = int(epoch_sec * fs)

    epochs = []

    for start in range(0, signal.shape[-1] - size, step):
        epochs.append(signal[:, start:start+size])

    return np.array(epochs)

In [5]:
def relative_band_power(epoch, fs):

    freqs, psd = welch(epoch, fs=fs, nperseg=fs*2, axis=-1)

    total_mask = (freqs >= 0.5) & (freqs <= 45)
    total_power = np.sum(psd[:, total_mask])

    features = []

    for band in BANDS.values():
        mask = (freqs >= band[0]) & (freqs < band[1])
        band_power = np.sum(psd[:, mask])
        features.append(band_power / total_power)

    return np.array(features)

In [6]:
def subject_to_features(eeg):

    eeg = bandpass_filter(eeg, FS, LOW, HIGH)

    epochs = epoch_signal(eeg, FS, EPOCH_SEC, OVERLAP)

    feats = []

    for ep in epochs:
        feats.append(relative_band_power(ep, FS))

    return np.array(feats)

In [7]:
def build_dataset(subjects, labels):

    X = []
    y = []
    groups = []

    for i, (eeg, label) in enumerate(zip(subjects, labels)):

        feats = subject_to_features(eeg)

        X.append(feats)
        y += [label] * len(feats)
        groups += [i] * len(feats)

    return np.vstack(X), np.array(y), np.array(groups)

In [8]:
# def evaluate_models(X, y, groups):

#     models = {
#         # "SVM": SVC(kernel="poly"),
#         "RF": RandomForestClassifier(),
#         # "kNN": KNeighborsClassifier(n_neighbors=3),
#         # "MLP": MLPClassifier(hidden_layer_sizes=(3,)),
#         # "LightGBM": LGBMClassifier(),
#     }

#     logo = LeaveOneGroupOut()

#     results = {}

#     for name, model in models.items():

#         y_true = []
#         y_pred = []

#         for train, test in logo.split(X, y, groups):

#             model.fit(X[train], y[train])
#             p = model.predict(X[test])

#             y_true.extend(y[test])
#             y_pred.extend(p)

#         acc = accuracy_score(y_true, y_pred)
#         f1 = f1_score(y_true, y_pred, average="weighted")

#         cm = confusion_matrix(y_true, y_pred)

#         results[name] = (acc, f1, cm)

#     return results

In [9]:
from scipy.stats import mode

def evaluate_models(X, y, groups):
    models = {
        # "SVM": SVC(kernel="poly"),
        "RF": RandomForestClassifier(n_jobs=8),
        "kNN": KNeighborsClassifier(n_neighbors=3, n_jobs=8),
        "logistic": LogisticRegression(n_jobs=8),
        # "MLP": MLPClassifier(hidden_layer_sizes=(3,)),
        # "LightGBM": LGBMClassifier(),
    }

    logo = LeaveOneGroupOut()
    results = {}

    for name, model in models.items():
        print(f"{name=}")
        subject_true = []
        subject_pred = []

        for train_idx, test_idx in logo.split(X, y, groups):
            model.fit(X[train_idx], y[train_idx])
            
            # Predict all epochs for the held-out subject
            epoch_preds = model.predict(X[test_idx])
            
            # Majority Vote: What does the model think this SUBJECT is?
            actual_label = y[test_idx][0]
            
            predicted_label = mode(epoch_preds)
            
            subject_true.append(actual_label)
            subject_pred.append(predicted_label[0])
            acc_fold = accuracy_score([actual_label], [predicted_label[0]])
            # print(f"accuracy of fold: {acc_fold}")

        acc = accuracy_score(subject_true, subject_pred)
        f1 = f1_score(subject_true, subject_pred, average="weighted")
        cm = confusion_matrix(subject_true, subject_pred)
        results[name] = (acc, f1, cm)

    return results

In [14]:
DATASET = "ds004504"
MASTER_TABLE_PATH = f"../metadata/{DATASET}/master_subject_table_{DATASET}.csv"

df = pd.read_csv(MASTER_TABLE_PATH)
df

,new_id,old_id,dataset_id,subject_id,session_id,session_num,run,age,sex,mmse,...,file_extension,file_path_raw,subject_dir,eeg_json_path,channels_tsv_path,bids_dataset_name,bids_version,bids_license,bids_dataset_doi,table_generated_at
0,1,sub-001,ds004504,sub-001,NaN,NaN,NaN,57,F,16,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-001,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:18
1,2,sub-002,ds004504,sub-002,NaN,NaN,NaN,78,F,22,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-002,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:18
2,3,sub-003,ds004504,sub-003,NaN,NaN,NaN,70,M,14,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-003,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:18
3,4,sub-004,ds004504,sub-004,NaN,NaN,NaN,67,F,20,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-004,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:18
4,5,sub-005,ds004504,sub-005,NaN,NaN,NaN,70,M,22,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-005,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-00...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,sub-084,ds004504,sub-084,NaN,NaN,NaN,71,F,24,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-084,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:24
84,85,sub-085,ds004504,sub-085,NaN,NaN,NaN,64,M,26,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-085,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:24
85,86,sub-086,ds004504,sub-086,NaN,NaN,NaN,49,M,26,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-086,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:24
86,87,sub-087,ds004504,sub-087,NaN,NaN,NaN,73,M,24,...,.set,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-087,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,/mnt/datasets_v7/gokul/EEGData/ds004504/sub-08...,A dataset of EEG recordings from: Alzheimer's ...,v1.2.1,CC0,doi:10.18112/openneuro.ds004504.v1.0.6,2026-03-04T00:06:24


In [15]:
df['file_path_raw'] = [
    path.replace('ds004504/', 'ds004504/cleaned/').replace('.set', '.fif') 
    for path in df['file_path_raw']
]
df["file_path_raw"][15]

'/mnt/datasets_v7/gokul/EEGData/ds004504/cleaned/sub-016/eeg/sub-016_task-eyesclosed_eeg.fif'

### alzheimer

In [16]:
import mne

subjects = []
labels = []

label_map = {'alzheimer': 0, 'frontotemporaldementia': 1, 'control': 2}
df['diagnosis_int'] = df['diagnosis'].map(label_map)

for index, row in df.iterrows():
    # raw = mne.io.read_raw_eeglab(df["file_path_raw"][index], preload=False, verbose='CRITICAL')
    raw = mne.io.read_raw_fif(df["file_path_raw"][index], preload=False, verbose='CRITICAL')
    subjects.append(raw.get_data())
    labels.append(df['diagnosis_int'][index])
    if(index==64):
        break

epoch level

In [14]:
# X, y, groups = build_dataset(subjects, labels)
# results = evaluate_models(X, y, groups)

# print(results)

subject level

In [18]:
X, y, groups = build_dataset(subjects, labels)
results = evaluate_models(X, y, groups)

print(results)

name='RF'
name='kNN'
name='logistic'
{'RF': (0.7384615384615385, 0.7392071933768298, array([[26, 10],
       [ 7, 22]])), 'kNN': (0.7538461538461538, 0.754546781491753, array([[26, 10],
       [ 6, 23]])), 'logistic': (0.8, 0.8, array([[26, 10],
       [ 3, 26]]))}


### Dementia

In [19]:
import mne

subjects = []
labels = []

label_map = {'alzheimer': 0, 'frontotemporaldementia': 1, 'control': 2}
df['diagnosis_int'] = df['diagnosis'].map(label_map)

for index, row in df.iterrows():
    if(index<=35):
        continue
    raw = mne.io.read_raw_eeglab(df["file_path_raw"][index], preload=False, verbose='CRITICAL')
    subjects.append(raw.get_data())
    labels.append(df['diagnosis_int'][index])
    

TypeError: MatFileReader.__init__() got an unexpected keyword argument 'uint16_codec'

In [18]:
X, y, groups = build_dataset(subjects, labels)
results = evaluate_models(X, y, groups)

print(results)

name='RF'
name='kNN'
name='logistic'
{'RF': (0.6730769230769231, 0.6613412228796844, array([[11, 12],
       [ 5, 24]])), 'kNN': (0.7115384615384616, 0.7011834319526629, array([[12, 11],
       [ 4, 25]])), 'logistic': (0.5961538461538461, 0.5038850038850039, array([[ 3, 20],
       [ 1, 28]]))}


### Multiclass

In [19]:
import mne

subjects = []
labels = []

label_map = {'alzheimer': 0, 'frontotemporaldementia': 1, 'control': 2}
df['diagnosis_int'] = df['diagnosis'].map(label_map)

for index, row in df.iterrows():
    raw = mne.io.read_raw_eeglab(df["file_path_raw"][index], preload=False, verbose='CRITICAL')
    subjects.append(raw.get_data())
    labels.append(df['diagnosis_int'][index])
    

In [20]:
X, y, groups = build_dataset(subjects, labels)
results = evaluate_models(X, y, groups)

print(results)

name='RF'
name='kNN'
name='logistic'


/mnt/scratch/gokul/ssl-eeg/code/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'RF': (0.5681818181818182, 0.5235806214159354, array([[29,  0,  7],
       [17,  3,  3],
       [11,  0, 18]])), 'kNN': (0.5340909090909091, 0.4675303202088916, array([[30,  0,  6],
       [19,  1,  3],
       [13,  0, 16]])), 'logistic': (0.5227272727272727, 0.44322007958371595, array([[30,  0,  6],
       [20,  0,  3],
       [13,  0, 16]]))}
